# Part 2: Exploring Classification Algorithms

In [710]:
import pandas as pd
from IPython.display import display
from IPython.display import Markdown
from sklearn.decomposition import PCA
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import r_regression
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import make_scorer
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import label_binarize
from sklearn.tree import DecisionTreeClassifier


In [711]:
def report(string):
    display(Markdown(string))

class Table:
    def tableHeader(self):
        doc = "|Algorithm| Hyperparameter | Value | accuracy | precision | recall | f1 |\n"
        doc += "| --- | --- | --- | --- |--- | --- | --- |\n"
        return doc

    def __init__(self):
        self.prev_algo = None
        self.prev_hyper = None
        self.prev_value = None
        self.doc = self.tableHeader()

    def tableCVScore(self, algorithm=None, hyperparameter=None, value=None, scores=None):
        rows = ""
        accuracy = scores['test_accuracy']
        precision = scores['test_precision']
        recall = scores['test_recall']
        f1 = scores['test_f1_score']
        out_algo = ""
        out_hyper = ""
        if (self.prev_algo != algorithm):
            self.prev_algo = algorithm
            out_algo = algorithm
            out_hyper = hyperparameter
        if (self.prev_hyper != hyperparameter):
            self.prev_hyper = hyperparameter
            out_hyper = hyperparameter
        rows += f"| {out_algo} | {out_hyper} | {value} | {accuracy:.4g} | {precision:.4g} | {recall:.4g} | {f1:.4g} |\n"
        self.doc = self.doc + rows
        return self

    def tableFooter(self):
        report(self.doc)


enable_debug = False

def debug(string):
    if (enable_debug):
        print(string)


# https://xkcd.com/221/ - 4 is overused
random_seed = 221

In [724]:
cross_validation_scores = {
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score, average='weighted'),
    'recall': make_scorer(recall_score, average='weighted'),
    'f1_score': make_scorer(f1_score, average='weighted')
}

def crossValidateUsingTree(random_state=random_seed, table=None, algorithm=None, hyperparameter=None, value=None, X=None, Y=None, max_depth=None,
                           cv=5):
    tree = DecisionTreeClassifier(criterion="entropy", random_state=random_state, max_depth=max_depth)
    cv_scores = cross_validate(tree, X, Y, cv=cv, scoring=cross_validation_scores)
    cv_scores_df = pd.DataFrame(cv_scores)
    table.tableCVScore(algorithm=algorithm, hyperparameter=hyperparameter, value=value, scores=cv_scores_df.mean())
    return cv_scores_df

def crossValidateUsingAdaBoost(random_state=random_seed, table=None, algorithm=None, hyperparameter=None, value=None, X=None, Y=None, n_estimators=50,
                               cv=5):
    tree = AdaBoostClassifier(random_state=random_state, n_estimators=n_estimators)
    cv_scores = cross_validate(tree, X, Y[target_column], cv=cv, scoring=cross_validation_scores)
    cv_scores_df = pd.DataFrame(cv_scores)
    table.tableCVScore(algorithm=algorithm, hyperparameter=hyperparameter, value=value, scores=cv_scores_df.mean())
    return cv_scores_df

def crossValidateUsingRandomForest(random_state=random_seed, table=None, algorithm=None, hyperparameter=None, value=None, X=None, Y=None, n_estimators=100,
                                   cv=5):
    tree = RandomForestClassifier(criterion="entropy", random_state=random_state, n_estimators=n_estimators)
    cv_scores = cross_validate(tree, X, Y[target_column], cv=cv, scoring=cross_validation_scores)
    cv_scores_df = pd.DataFrame(cv_scores)
    table.tableCVScore(algorithm=algorithm, hyperparameter=hyperparameter, value=value, scores=cv_scores_df.mean())
    return cv_scores_df

def crossValidateUsingKnn(table=None, X=None, Y=None, algorithm=None, hyperparameter=None, value=None, n_neighbours=5,
                          cv=5):
    knn = KNeighborsClassifier(n_neighbors=n_neighbours)
    cv_scores = cross_validate(knn, X, Y[target_column], cv=cv, scoring=cross_validation_scores)
    cv_scores_df = pd.DataFrame(cv_scores)
    table.tableCVScore(algorithm=algorithm, hyperparameter=hyperparameter, value=value, scores=cv_scores_df.mean())
    return cv_scores_df

def scores(Y, Y_pred):
    return {
        'test_accuracy': accuracy_score(Y, Y_pred),
        'test_precision': precision_score(Y, Y_pred, average='weighted'),
        'test_recall': recall_score(Y, Y_pred, average='weighted'),
        'test_f1_score': f1_score(Y, Y_pred, average='weighted'),
    }

def trainAndPredictDecisionTree(random_state=random_seed, table=None, algorithm=None, hyperparameter=None, value=None, X=None, Y=None, max_depth=None, testX=None, testY=None):
    tree = DecisionTreeClassifier(criterion="entropy", random_state=random_state, max_depth=max_depth)
    tree.fit(X, Y)
    y_pred = tree.predict(testX)
    table.tableCVScore(algorithm=algorithm, hyperparameter=hyperparameter, value=value, scores=scores(testY, y_pred))
    return table

def trainAndPredictAdaBoost(random_state=random_seed, table=None, algorithm=None, hyperparameter=None, value=None, X=None, Y=None, n_estimators=50, testX=None, testY=None):
    tree = AdaBoostClassifier(random_state=random_state, n_estimators=n_estimators)
    tree.fit(X,Y[target_column])
    y_pred = tree.predict(testX)
    table.tableCVScore(algorithm=algorithm, hyperparameter=hyperparameter, value=value, scores=scores(testY, y_pred))
    return table

def trainAndPredictRandomForest(random_state=random_seed, table=None, algorithm=None, hyperparameter=None, value=None, X=None, Y=None, n_estimators=100, testX=None, testY=None):
    tree = RandomForestClassifier(criterion="entropy", random_state=random_state, n_estimators=n_estimators)
    tree.fit(X, Y[target_column])
    y_pred = tree.predict(testX)
    table.tableCVScore(algorithm=algorithm, hyperparameter=hyperparameter, value=value, scores=scores(testY, y_pred))
    return table

def trainAndPredictKnn(table=None, X=None, Y=None, algorithm=None, hyperparameter=None, value=None, n_neighbours=5, testX=None, testY=None):
    knn = KNeighborsClassifier(n_neighbors=n_neighbours)
    knn.fit(X, Y[target_column])
    y_pred = knn.predict(testX)
    table.tableCVScore(algorithm=algorithm, hyperparameter=hyperparameter, value=value, scores=scores(testY, y_pred))
    return table


## Data Loading

In [725]:
df_csv = pd.read_csv('data/breast-cancer.csv')

target_column = 'diagnosis'
id_column = 'id'

feature_names = df_csv.columns.tolist()
debug(feature_names)
df_csv = df_csv.drop(id_column, axis=1)

Y_df = df_csv[[target_column]]
X_df = df_csv.drop([target_column], axis=1)
# unique converts to an NDArray, so the sort needs to happen first.
target_columns = Y_df.columns.tolist()
debug(f"target column: {target_columns}")
feature_names = X_df.columns.tolist()
debug(f"feature names: {feature_names}")
labels = Y_df[target_column].sort_values().unique()
debug(f"labels: {labels}")

# Split the data into Train and Test
X_train, X_test, Y_train, Y_test = train_test_split(
    X_df, Y_df, test_size= 0.3, random_state=random_seed)

## 1. Missing Data Processing and Data Normalization

In [726]:
mean_imputer = SimpleImputer(missing_values=0, strategy='mean')
mean_imputer.fit(X_train)
X_train_mean = mean_imputer.transform(X_train)
X_test_mean = mean_imputer.transform(X_test)
X_df_mean = mean_imputer.transform(X_df)

median_imputer = SimpleImputer(missing_values=0, strategy='median')
median_imputer.fit(X_train)
X_train_median = median_imputer.transform(X_train)
X_test_median = median_imputer.transform(X_test)
X_df_median = median_imputer.transform(X_df)

## 1.a DecisionTree evaluation of imputation methods

In [727]:
table = Table()

parameters = {
  "table":table, "algorithm": "Decision Tree", "hyperparameter": "imputation"
}
crossValidateUsingTree(**parameters, value="baseline", X=X_train, Y=Y_train)
crossValidateUsingTree(**parameters, value="mean", X=X_train_mean, Y=Y_train)
crossValidateUsingTree(**parameters, value="median", X=X_train_median, Y=Y_train)
crossValidateUsingTree(**parameters, value="full mean", X=X_df_mean, Y=Y_df)
crossValidateUsingTree(**parameters, value="full median", X=X_df_median, Y=Y_df)

table.tableFooter()


|Algorithm| Hyperparameter | Value | accuracy | precision | recall | f1 |
| --- | --- | --- | --- |--- | --- | --- |
| Decision Tree | imputation | baseline | 0.9398 | 0.9419 | 0.9398 | 0.9396 |
|  |  | mean | 0.9323 | 0.9358 | 0.9323 | 0.9323 |
|  |  | median | 0.9323 | 0.9358 | 0.9323 | 0.9323 |
|  |  | full mean | 0.9244 | 0.9276 | 0.9244 | 0.9242 |
|  |  | full median | 0.9262 | 0.929 | 0.9262 | 0.9259 |


### Evaluation of imputation methods

For evaluation, the evaluation metric was set to "entropy" for information gain, and max_depth is None to give the tree the best chance at success. Decision Trees are very tolerant of missing data. When using cross validation, both median and mean imputation produced a tree that performed worse than baseline.

Adding in median and mean cross validation using the full dataset, we see that median works slightly better than mean, which for a decision tree would make sense. It puts the missing value in the middle of the data, ignoring outliers.

In [728]:
X_train = X_train_median
X_test = X_test_median
baseline_cv_scores_df = median_imputation_cv_scores_df
baseline_cv_scores_dev_df = median_imputation_cv_scores_dev_df

## 1.b Standardization

In [729]:
standard_scaler = StandardScaler()
standard_scaler.fit(X_train)
X_train_standardized = standard_scaler.transform(X_train, copy=True)
X_test_standardized = standard_scaler.transform(X_test, copy=True)
X_df_standardized = standard_scaler.transform(X_df_median, copy=True)

min_max_scaler = MinMaxScaler(copy=True) # don't clip. Why is MinMaxScaler different to StandardizedScaler?!?
min_max_scaler.fit(X_train)
X_train_min_max = min_max_scaler.transform(X_train)
X_test_min_max = min_max_scaler.transform(X_test)
X_df_min_max = min_max_scaler.transform(X_df_median)


In [730]:
table = Table()

parameters = {
  "algorithm":"Decision Tree", "hyperparameter":"normalization"
}
table.tableCVScore(**parameters, value="baseline", scores=baseline_cv_scores_df)
parameters["table"] = table
crossValidateUsingTree(**parameters, value="standardized", X=X_train_standardized, Y=Y_train)
crossValidateUsingTree(**parameters, value="min_max", X=X_train_min_max, Y=Y_train)
crossValidateUsingTree(**parameters, value="full standardized", X=X_df_standardized, Y=Y_df)
crossValidateUsingTree(**parameters, value="full min_max", X=X_df_min_max, Y=Y_df)

parameters = {
  "table":table, "algorithm": "KNN", "hyperparameter": "normalization"
}
crossValidateUsingKnn(**parameters, value="baseline", X=X_train, Y=Y_train)
crossValidateUsingKnn(**parameters, value="standardized", X=X_train_standardized, Y=Y_train)
crossValidateUsingKnn(**parameters, value="min-max", X=X_train_min_max, Y=Y_train)
crossValidateUsingKnn(**parameters, value="full standardized", X=X_df_standardized, Y=Y_df)
crossValidateUsingKnn(**parameters, value="full min-max", X=X_df_min_max, Y=Y_df)

table.tableFooter()

|Algorithm| Hyperparameter | Value | accuracy | precision | recall | f1 |
| --- | --- | --- | --- |--- | --- | --- |
| Decision Tree | normalization | baseline | 0.9323 | 0.9358 | 0.9323 | 0.9323 |
|  |  | standardized | 0.9323 | 0.9358 | 0.9323 | 0.9323 |
|  |  | min_max | 0.9323 | 0.9358 | 0.9323 | 0.9323 |
|  |  | full standardized | 0.9262 | 0.929 | 0.9262 | 0.9259 |
|  |  | full min_max | 0.9262 | 0.929 | 0.9262 | 0.9259 |
| KNN | normalization | baseline | 0.9373 | 0.9385 | 0.9373 | 0.9362 |
|  |  | standardized | 0.9598 | 0.9614 | 0.9598 | 0.9593 |
|  |  | min-max | 0.9624 | 0.9637 | 0.9624 | 0.9619 |
|  |  | full standardized | 0.9666 | 0.9675 | 0.9666 | 0.9664 |
|  |  | full min-max | 0.9701 | 0.9706 | 0.9701 | 0.97 |


### Evaluation of normalization methods

There is no benefit to standardization over baseline in decision trees for either method. This makes sense because the tree is splitting at a threshold, so the number of thresholds doesn't change with scale changes.

KNN shows an improvement using min-max, and this continues through to the full dataset. This implies that the data isn't normally distributed, with multiple modes. Standardization provides additional bits to the data values close to the mean. MinMax scaling provides the same number of bits over the address range. If there are multiple modes at either end of the distribution, MinMax will allow them to be more easily differentiated, where standardization might lose too many digits.

In [731]:
X_train = X_train_min_max
X_test = X_test_min_max

## 2. Classifier Exploration and Hyperparameter Tuning

In [732]:
table = Table()

parameters = {
  "table":table, "algorithm": "KNN", "hyperparameter": "n_neighbours",
  "X":X_train, "Y":Y_train, "testX" : X_test, "testY": Y_test
}
knn_neighbours = [3,9,15,21]
for n in knn_neighbours:
    trainAndPredictKnn(**parameters, value=n, n_neighbours=n)

parameters = {
    "table":table, "algorithm": "Decision Tree", "hyperparameter": "max_depth",
    "X":X_train, "Y":Y_train, "testX" : X_test, "testY": Y_test
}
decision_tree_depths = [2,8,14,None]
for n in decision_tree_depths:
    trainAndPredictDecisionTree(**parameters, value=n, max_depth=n)

parameters = {
    "table":table, "algorithm": "AdaBoost", "hyperparameter": "n_estimators",
    "X":X_train, "Y":Y_train, "testX" : X_test, "testY": Y_test
}
n_estimators = [10,20,30,100]
for n in n_estimators:
    trainAndPredictAdaBoost(**parameters, value=n, n_estimators=n)

parameters = {
    "table":table, "algorithm": "Random Forest", "hyperparameter": "n_estimators",
    "X":X_train, "Y":Y_train, "testX" : X_test, "testY": Y_test
}
n_estimators = [10,30,50,60,100]
for n in n_estimators:
    trainAndPredictRandomForest(**parameters, value=n, n_estimators=n)

table.tableFooter()

|Algorithm| Hyperparameter | Value | accuracy | precision | recall | f1 |
| --- | --- | --- | --- |--- | --- | --- |
| KNN | n_neighbours | 3 | 0.9649 | 0.9668 | 0.9649 | 0.9646 |
|  |  | 9 | 0.9708 | 0.9721 | 0.9708 | 0.9705 |
|  |  | 15 | 0.9708 | 0.9721 | 0.9708 | 0.9705 |
|  |  | 21 | 0.9532 | 0.9566 | 0.9532 | 0.9526 |
| Decision Tree | max_depth | 2 | 0.883 | 0.8841 | 0.883 | 0.8815 |
|  |  | 8 | 0.9415 | 0.9445 | 0.9415 | 0.9407 |
|  |  | 14 | 0.9415 | 0.9445 | 0.9415 | 0.9407 |
|  |  | None | 0.9415 | 0.9445 | 0.9415 | 0.9407 |
| AdaBoost | n_estimators | 10 | 0.9591 | 0.9602 | 0.9591 | 0.9588 |
|  |  | 20 | 0.9532 | 0.9549 | 0.9532 | 0.9528 |
|  |  | 30 | 0.9649 | 0.9656 | 0.9649 | 0.9647 |
|  |  | 100 | 0.9649 | 0.9656 | 0.9649 | 0.9647 |
| Random Forest | n_estimators | 10 | 0.9298 | 0.931 | 0.9298 | 0.9292 |
|  |  | 30 | 0.9474 | 0.9483 | 0.9474 | 0.947 |
|  |  | 50 | 0.9649 | 0.9668 | 0.9649 | 0.9646 |
|  |  | 60 | 0.9591 | 0.9616 | 0.9591 | 0.9586 |
|  |  | 100 | 0.9532 | 0.9566 | 0.9532 | 0.9526 |


### Observe and discuss the impact of varying the hyperparameter

This felt a little dirty, evaluating the hyperparameter against the test data. We can explain the results afterwards, but predicting them is difficult. All algorithms other than KNN increased in performance as the hyperparameter was increased. This makes sense as the non-KNN algorithms will add more information as the parameter is increased. I added N=100 and max_depth=None to see if there were elbows where performance dropped off. Changing the hyperparameters had very small impacts on overall performance, less than 1% for KNN, 2% for AdaBoost, 3.5% for Random Forest. Decision Tree showed the largest swing at 6%.

AdaBoost performed best at N=30. AdaBoost learns which instances are classified improperly, weighting them in the next iteration. As more iterations are performed, the error term will naturally decrease. The algorithm is able to continue to decrease the error as more cpu is expended. However, each iteration will also add another decision tree to the final model.

RandomForest was second best with n = 50, outperforming the regular decision tree. The random subset used for each tree allow features which are unimportant to have a chance to be ignored. However, as the number of trees increases, performance dropped back down.

Decision Tree hit peak performance at max_depth=8. This is likely because there were no further splits in the training data. Given the 539 instances, a 70% training split = 377, which is smaller than the size of a fully populated tree at 2^8 * 2 = 512 nodes. The minimum leaf size won't allow more nodes. Decision Trees do seem to have a minimum required number of nodes, somwhere between 10 and 20 for this dataset.

KNN peaked at n = 3. Increasing the number of neighbours decreased performance. This is likely because the correct n to use is related to the number of instances in the dataset, and the number of expected clusters.

This shows that algorithms that are able to improve their error term with additional iterations will outperform algorithms that increase the amount of data considered.

## 3. Feature Engineering and Dimensionality Reduction

### 3.a Pearson Correlation

In [733]:
# Convert back to DataFrame so we can see what's what.
report("**Step 1 - use normalised data**")
X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df = pd.DataFrame(X_test, columns=feature_names)

report("**Step 2 - translate Y to binary**")
Y_train_transformed = label_binarize(Y_train[target_column].tolist(), classes=['B', 'M']).ravel()
Y_test_transformed = label_binarize(Y_test[target_column].tolist(), classes=['B', 'M']).ravel()

report("**Step 3 - run the regression**")
pearson = r_regression(X_train, Y_train_transformed)

report("**Step 4 - Iterate over the list, and find the indices that are > 0.6**")
X_train_filtered = X_train_df.copy()
X_test_filtered = X_test_df.copy()
threshold = 0.6

pearson_filtered_columns = []
for i in range(0, len(pearson)):
    r = pearson[i]
    if r >= threshold:
        pearson_filtered_columns.append(i)

X_train_filtered = X_train_filtered.iloc[:,pearson_filtered_columns]
X_test_filtered = X_test_filtered.iloc[:,pearson_filtered_columns]

report("**Step 5 - Produce output for report**")
report("**Step 5.a - Generate the 2d correlation matrix using Pandas - otherwise unused**")
X_correlation = X_train_df.copy()
X_correlation[target_column] = Y_train_transformed

two_d_correlation = X_correlation.corr(method="pearson")
report("**Correlation Matrix:**")
report(two_d_correlation.to_markdown())

report(f"**Number of original features:** {len(feature_names)}")
report(f"**Number retained features:** {len(X_train_filtered.columns)}")

**Step 1 - use normalised data**

**Step 2 - translate Y to binary**

**Step 3 - run the regression**

**Step 4 - Iterate over the list, and find the indices that are > 0.6**

**Step 5 - Produce output for report**

**Step 5.a - Generate the 2d correlation matrix using Pandas - otherwise unused**

**Correlation Matrix:**

|                         |   radius_mean |   texture_mean |   perimeter_mean |   area_mean |   smoothness_mean |   compactness_mean |   concavity_mean |   concave points_mean |   symmetry_mean |   fractal_dimension_mean |   radius_se |   texture_se |   perimeter_se |    area_se |   smoothness_se |   compactness_se |   concavity_se |   concave points_se |   symmetry_se |   fractal_dimension_se |   radius_worst |   texture_worst |   perimeter_worst |   area_worst |   smoothness_worst |   compactness_worst |   concavity_worst |   concave points_worst |   symmetry_worst |   fractal_dimension_worst |    diagnosis |
|:------------------------|--------------:|---------------:|-----------------:|------------:|------------------:|-------------------:|-----------------:|----------------------:|----------------:|-------------------------:|------------:|-------------:|---------------:|-----------:|----------------:|-----------------:|---------------:|--------------------:|--------------:|-----------------------:|---------------:|----------------:|------------------:|-------------:|-------------------:|--------------------:|------------------:|-----------------------:|-----------------:|--------------------------:|-------------:|
| radius_mean             |     1         |     0.385672   |        0.997861  |   0.988006  |         0.158837  |          0.528282  |         0.653981 |              0.812538 |       0.171622  |              -0.306374   |  0.70014    |  -0.0841067  |      0.696177  |  0.758164  |      -0.196311  |         0.209899 |      0.146951  |           0.316315  |  -0.11821     |             -0.0550476 |      0.972592  |       0.361168  |         0.967246  |    0.943017  |        0.146304    |           0.451614  |        0.526159   |              0.736021  |        0.195858  |                 0.0431111 |  0.743086    |
| texture_mean            |     0.385672  |     1          |        0.39054   |   0.382414  |        -0.0463297 |          0.263987  |         0.335993 |              0.345599 |       0.0815752 |              -0.109117   |  0.328933   |   0.357636   |      0.333682  |  0.32219   |      -0.0303147 |         0.1927   |      0.126985  |           0.175217  |  -0.00282341  |              0.0311163 |      0.412253  |       0.917975  |         0.415498  |    0.401916  |        0.0495641   |           0.316329  |        0.333596   |              0.346451  |        0.121702  |                 0.124447  |  0.458916    |
| perimeter_mean          |     0.997861  |     0.39054    |        1         |   0.987623  |         0.194833  |          0.578168  |         0.693394 |              0.84097  |       0.204995  |              -0.256289   |  0.713057   |  -0.0742319  |      0.716424  |  0.768803  |      -0.175564  |         0.254154 |      0.180138  |           0.34762   |  -0.0944147   |             -0.019312  |      0.972633  |       0.366324  |         0.972804  |    0.944365  |        0.175366    |           0.49363   |        0.563128   |              0.762938  |        0.219139  |                 0.087509  |  0.753717    |
| area_mean               |     0.988006  |     0.382414   |        0.987623  |   1         |         0.166979  |          0.526744  |         0.670172 |              0.821018 |       0.178461  |              -0.273857   |  0.749339   |  -0.0553505  |      0.744365  |  0.817489  |      -0.146758  |         0.22266  |      0.167878  |           0.333401  |  -0.0935633   |             -0.0273539 |      0.969598  |       0.351668  |         0.96574   |    0.965043  |        0.153385    |           0.43461   |        0.522222   |              0.729972  |        0.178855  |                 0.0437391 |  0.726408    |
| smoothness_mean         |     0.158837  |    -0.0463297  |        0.194833  |   0.166979  |         1         |          0.638484  |         0.494225 |              0.531128 |       0.575932  |               0.579446   |  0.299155   |   0.0856847  |      0.306834  |  0.244289  |       0.317465  |         0.306293 |      0.231197  |           0.35726   |   0.215424    |              0.247155  |      0.198635  |       0.0238485 |         0.228096  |    0.195246  |        0.793623    |           0.453287  |        0.406796   |              0.472538  |        0.388518  |                 0.471404  |  0.334149    |
| compactness_mean        |     0.528282  |     0.263987   |        0.578168  |   0.526744  |         0.638484  |          1         |         0.877742 |              0.833354 |       0.590668  |               0.54796    |  0.531608   |   0.0561717  |      0.598757  |  0.503419  |       0.14015   |         0.728854 |      0.538224  |           0.617623  |   0.241526    |              0.472469  |      0.552533  |       0.275378  |         0.609798  |    0.533317  |        0.536742    |           0.876193  |        0.813777   |              0.809613  |        0.495974  |                 0.694667  |  0.586917    |
| concavity_mean          |     0.653981  |     0.335993   |        0.693394  |   0.670172  |         0.494225  |          0.877742  |         1        |              0.913781 |       0.516928  |               0.359641   |  0.657668   |   0.132137   |      0.695729  |  0.64122   |       0.141357  |         0.678809 |      0.68952   |           0.696493  |   0.217388    |              0.471016  |      0.666993  |       0.322758  |         0.710544  |    0.662856  |        0.420197    |           0.753033  |        0.892765   |              0.855936  |        0.400916  |                 0.534232  |  0.673717    |
| concave points_mean     |     0.812538  |     0.345599   |        0.84097   |   0.821018  |         0.531128  |          0.833354  |         0.913781 |              1        |       0.472049  |               0.1787     |  0.730783   |   0.070102   |      0.75315   |  0.723923  |       0.05574   |         0.486345 |      0.409581  |           0.586126  |   0.111328    |              0.248904  |      0.823476  |       0.339282  |         0.850457  |    0.808992  |        0.44832     |           0.682338  |        0.763173   |              0.914097  |        0.374096  |                 0.395874  |  0.77093     |
| symmetry_mean           |     0.171622  |     0.0815752  |        0.204995  |   0.178461  |         0.575932  |          0.590668  |         0.516928 |              0.472049 |       1         |               0.449868   |  0.32242    |   0.119397   |      0.330654  |  0.263434  |       0.16852   |         0.403328 |      0.344064  |           0.360443  |   0.447424    |              0.310869  |      0.202921  |       0.095311  |         0.233498  |    0.199881  |        0.417544    |           0.477087  |        0.457673   |              0.429448  |        0.693076  |                 0.437422  |  0.330513    |
| fractal_dimension_mean  |    -0.306374  |    -0.109117   |       -0.256289  |  -0.273857  |         0.579446  |          0.54796   |         0.359641 |              0.1787   |       0.449868  |               1          |  0.00368633 |   0.18719    |      0.0552314 | -0.0851035 |       0.369454  |         0.552815 |      0.479089  |           0.399164  |   0.351707    |              0.680414  |     -0.252724  |      -0.0791681 |        -0.199824  |   -0.223043  |        0.445534    |           0.432467  |        0.350958   |              0.187451  |        0.280653  |                 0.750865  | -0.0454787   |
| radius_se               |     0.70014   |     0.328933   |        0.713057  |   0.749339  |         0.299155  |          0.531608  |         0.657668 |              0.730783 |       0.32242   |               0.00368633 |  1          |   0.222548   |      0.972339  |  0.94643   |       0.167481  |         0.388218 |      0.342976  |           0.549859  |   0.200301    |              0.224051  |      0.745901  |       0.253024  |         0.752155  |    0.784903  |        0.17449     |           0.344581  |        0.439314   |              0.584027  |        0.128994  |                 0.079359  |  0.613725    |
| texture_se              |    -0.0841067 |     0.357636   |       -0.0742319 |  -0.0553505 |         0.0856847 |          0.0561717 |         0.132137 |              0.070102 |       0.119397  |               0.18719    |  0.222548   |   1          |      0.217742  |  0.119887  |       0.384286  |         0.258175 |      0.24766   |           0.301443  |   0.398394    |              0.299654  |     -0.0964727 |       0.381314  |        -0.0887929 |   -0.0669328 |       -0.0695574   |          -0.0730234 |       -0.00554378 |             -0.0476679 |       -0.142408  |                -0.0401475 |  0.00881065  |
| perimeter_se            |     0.696177  |     0.333682   |        0.716424  |   0.744365  |         0.306834  |          0.598757  |         0.695729 |              0.75315  |       0.330654  |               0.0552314  |  0.972339   |   0.217742   |      1         |  0.931758  |       0.160539  |         0.457756 |      0.367521  |           0.588522  |   0.221971    |              0.24556   |      0.734546  |       0.258521  |         0.76123   |    0.772747  |        0.175674    |           0.415783  |        0.487539   |              0.617624  |        0.14425   |                 0.132685  |  0.601874    |
| area_se                 |     0.758164  |     0.32219    |        0.768803  |   0.817489  |         0.244289  |          0.503419  |         0.64122  |              0.723923 |       0.263434  |              -0.0851035  |  0.94643    |   0.119887   |      0.931758  |  1         |       0.0691034 |         0.323599 |      0.267863  |           0.439847  |   0.0875827   |              0.123413  |      0.796052  |       0.266286  |         0.802785  |    0.854983  |        0.16404     |           0.35487   |        0.442875   |              0.594352  |        0.129236  |                 0.0585832 |  0.591803    |
| smoothness_se           |    -0.196311  |    -0.0303147  |       -0.175564  |  -0.146758  |         0.317465  |          0.14015   |         0.141357 |              0.05574  |       0.16852   |               0.369454   |  0.167481   |   0.384286   |      0.160539  |  0.0691034 |       1         |         0.394417 |      0.322741  |           0.42369   |   0.413404    |              0.45247   |     -0.206433  |      -0.105687  |        -0.189559  |   -0.16204   |        0.289759    |          -0.0460826 |       -0.0181206  |             -0.0610216 |       -0.12114   |                 0.0614882 | -0.0367813   |
| compactness_se          |     0.209899  |     0.1927     |        0.254154  |   0.22266   |         0.306293  |          0.728854  |         0.678809 |              0.486345 |       0.403328  |               0.552815   |  0.388218   |   0.258175   |      0.457756  |  0.323599  |       0.394417  |         1        |      0.787635  |           0.761109  |   0.410205    |              0.793482  |      0.201476  |       0.138891  |         0.258676  |    0.203928  |        0.190549    |           0.661239  |        0.628893   |              0.461897  |        0.239942  |                 0.581663  |  0.279967    |
| concavity_se            |     0.146951  |     0.126985   |        0.180138  |   0.167878  |         0.231197  |          0.538224  |         0.68952  |              0.409581 |       0.344064  |               0.479089   |  0.342976   |   0.24766    |      0.367521  |  0.267863  |       0.322741  |         0.787635 |      1         |           0.800704  |   0.358901    |              0.764621  |      0.142019  |       0.0731476 |         0.180282  |    0.153186  |        0.125034    |           0.443257  |        0.649931   |              0.402687  |        0.17513   |                 0.442261  |  0.215087    |
| concave points_se       |     0.316315  |     0.175217   |        0.34762   |   0.333401  |         0.35726   |          0.617623  |         0.696493 |              0.586126 |       0.360443  |               0.399164   |  0.549859   |   0.301443   |      0.588522  |  0.439847  |       0.42369   |         0.761109 |      0.800704  |           1         |   0.35999     |              0.666194  |      0.303485  |       0.0823569 |         0.340027  |    0.308287  |        0.185342    |           0.438263  |        0.566469   |              0.56029   |        0.0933982 |                 0.342935  |  0.37854     |
| symmetry_se             |    -0.11821   |    -0.00282341 |       -0.0944147 |  -0.0935633 |         0.215424  |          0.241526  |         0.217388 |              0.111328 |       0.447424  |               0.351707   |  0.200301   |   0.398394   |      0.221971  |  0.0875827 |       0.413404  |         0.410205 |      0.358901  |           0.35999   |   1           |              0.364367  |     -0.132295  |      -0.0768584 |        -0.106614  |   -0.113788  |       -1.35944e-05 |           0.0824748 |        0.111042   |              0.0130819 |        0.402317  |                 0.0880276 | -0.000854346 |
| fractal_dimension_se    |    -0.0550476 |     0.0311163  |       -0.019312  |  -0.0273539 |         0.247155  |          0.472469  |         0.471016 |              0.248904 |       0.310869  |               0.680414   |  0.224051   |   0.299654   |      0.24556   |  0.123413  |       0.45247   |         0.793482 |      0.764621  |           0.666194  |   0.364367    |              1         |     -0.0540215 |      -0.0261338 |        -0.0160523 |   -0.0315194 |        0.103384    |           0.35518   |        0.385773   |              0.207051  |        0.0716195 |                 0.570349  |  0.0559724   |
| radius_worst            |     0.972592  |     0.412253   |        0.972633  |   0.969598  |         0.198635  |          0.552533  |         0.666993 |              0.823476 |       0.202921  |              -0.252724   |  0.745901   |  -0.0964727  |      0.734546  |  0.796052  |      -0.206433  |         0.201476 |      0.142019  |           0.303485  |  -0.132295    |             -0.0540215 |      1         |       0.419014  |         0.993703  |    0.983902  |        0.237263    |           0.504591  |        0.569453   |              0.780081  |        0.267261  |                 0.122035  |  0.786105    |
| texture_worst           |     0.361168  |     0.917975   |        0.366324  |   0.351668  |         0.0238485 |          0.275378  |         0.322758 |              0.339282 |       0.095311  |              -0.0791681  |  0.253024   |   0.381314   |      0.258521  |  0.266286  |      -0.105687  |         0.138891 |      0.0731476 |           0.0823569 |  -0.0768584   |             -0.0261338 |      0.419014  |       1         |         0.422239  |    0.403365  |        0.209246    |           0.391655  |        0.381958   |              0.40019   |        0.243027  |                 0.225949  |  0.492398    |
| perimeter_worst         |     0.967246  |     0.415498   |        0.972804  |   0.96574   |         0.228096  |          0.609798  |         0.710544 |              0.850457 |       0.233498  |              -0.199824   |  0.752155   |  -0.0887929  |      0.76123   |  0.802785  |      -0.189559  |         0.258676 |      0.180282  |           0.340027  |  -0.106614    |             -0.0160523 |      0.993703  |       0.422239  |         1         |    0.978725  |        0.259314    |           0.56116   |        0.615655   |              0.809614  |        0.290468  |                 0.173458  |  0.788884    |
| area_worst              |     0.943017  |     0.401916   |        0.944365  |   0.965043  |         0.195246  |          0.533317  |         0.662856 |              0.808992 |       0.199881  |              -0.223043   |  0.784903   |  -0.0669328  |      0.772747  |  0.854983  |      -0.16204   |         0.203928 |      0.153186  |           0.308287  |  -0.113788    |             -0.0315194 |      0.983902  |       0.403365  |         0.978725  |    1         |        0.23086     |           0.471992  |        0.547912   |              0.752815  |        0.237098  |                 0.112595  |  0.745702    |
| smoothness_worst        |     0.146304  |     0.0495641  |        0.175366  |   0.153385  |         0.793623  |          0.536742  |         0.420197 |              0.44832  |       0.417544  |               0.445534   |  0.17449    |  -0.0695574  |      0.175674  |  0.16404   |       0.289759  |         0.190549 |      0.125034  |           0.185342  |  -1.35944e-05 |              0.103384  |      0.237263  |       0.209246  |         0.259314  |    0.23086   |        1           |           0.544226  |        0.479058   |              0.535571  |        0.479605  |                 0.563697  |  0.41142     |
| compactness_worst       |     0.451614  |     0.316329   |        0.49363   |   0.43461   |         0.453287  |          0.876193  |         0.753033 |              0.682338 |       0.477087  |               0.432467   |  0.344581   |  -0.0730234  |      0.415783  |  0.35487   |      -0.0460826 |         0.661239 |      0.443257  |           0.438263  |   0.0824748   |              0.35518   |      0.504591  |       0.391655  |         0.56116   |    0.471992  |        0.544226    |           1         |        0.875607   |              0.801871  |        0.599656  |                 0.806996  |  0.585649    |
| concavity_worst         |     0.526159  |     0.333596   |        0.563128  |   0.522222  |         0.406796  |          0.813777  |         0.892765 |              0.763173 |       0.457673  |               0.350958   |  0.439314   |  -0.00554378 |      0.487539  |  0.442875  |      -0.0181206 |         0.628893 |      0.649931  |           0.566469  |   0.111042    |              0.385773  |      0.569453  |       0.381958  |         0.615655  |    0.547912  |        0.479058    |           0.875607  |        1          |              0.859954  |        0.527986  |                 0.684484  |  0.644021    |
| concave points_worst    |     0.736021  |     0.346451   |        0.762938  |   0.729972  |         0.472538  |          0.809613  |         0.855936 |              0.914097 |       0.429448  |               0.187451   |  0.584027   |  -0.0476679  |      0.617624  |  0.594352  |      -0.0610216 |         0.461897 |      0.402687  |           0.56029   |   0.0130819   |              0.207051  |      0.780081  |       0.40019   |         0.809614  |    0.752815  |        0.535571    |           0.801871  |        0.859954   |              1         |        0.490644  |                 0.539145  |  0.781459    |
| symmetry_worst          |     0.195858  |     0.121702   |        0.219139  |   0.178855  |         0.388518  |          0.495974  |         0.400916 |              0.374096 |       0.693076  |               0.280653   |  0.128994   |  -0.142408   |      0.14425   |  0.129236  |      -0.12114   |         0.239942 |      0.17513   |           0.0933982 |   0.402317    |              0.0716195 |      0.267261  |       0.243027  |         0.290468  |    0.237098  |        0.479605    |           0.599656  |        0.527986   |              0.490644  |        1         |                 0.524224  |  0.40802     |
| fractal_dimension_worst |     0.0431111 |     0.124447   |        0.087509  |   0.0437391 |         0.471404  |          0.694667  |         0.534232 |              0.395874 |       0.437422  |               0.750865   |  0.079359   |  -0.0401475  |      0.132685  |  0.0585832 |       0.0614882 |         0.581663 |      0.442261  |           0.342935  |   0.0880276   |              0.570349  |      0.122035  |       0.225949  |         0.173458  |    0.112595  |        0.563697    |           0.806996  |        0.684484   |              0.539145  |        0.524224  |                 1         |  0.314269    |
| diagnosis               |     0.743086  |     0.458916   |        0.753717  |   0.726408  |         0.334149  |          0.586917  |         0.673717 |              0.77093  |       0.330513  |              -0.0454787  |  0.613725   |   0.00881065 |      0.601874  |  0.591803  |      -0.0367813 |         0.279967 |      0.215087  |           0.37854   |  -0.000854346 |              0.0559724 |      0.786105  |       0.492398  |         0.788884  |    0.745702  |        0.41142     |           0.585649  |        0.644021   |              0.781459  |        0.40802   |                 0.314269  |  1           |

**Number of original features:** 30

**Number retained features:** 12

In [734]:
# Train a decision tree using X_train_filtered and X_test_filtered
table = Table()

parameters = {
  "table":table, "algorithm": "Decision Tree", "hyperparameter": "feature selection",
  "X":X_train, "Y":Y_train, "testX" : X_test, "testY": Y_test
}
trainAndPredictDecisionTree(**parameters, value="baseline", max_depth=None)

parameters = {
  "table":table, "algorithm": "Decision Tree", "hyperparameter": "feature selection",
  "X":X_train_filtered, "Y":Y_train, "testX" : X_test_filtered, "testY": Y_test
}
trainAndPredictDecisionTree(**parameters, value="pearson", max_depth=None)

table.tableFooter()

|Algorithm| Hyperparameter | Value | accuracy | precision | recall | f1 |
| --- | --- | --- | --- |--- | --- | --- |
| Decision Tree | feature selection | baseline | 0.9415 | 0.9445 | 0.9415 | 0.9407 |
|  |  | pearson | 0.9415 | 0.9419 | 0.9415 | 0.9412 |


#### Pearson Correlation Performance Evaluation

Using Pearson Correlation for feature selection reduced the number of features from 30 to 12. It had minimal impact on the resulting performance. The drop in precision was a single mis-predicted positive. Looking at the depth of the resulting trees, the baseline tree had a max depth of 5, while the filtered tree had a depth of 8. A shallower tree will perform better - fewer operations to classify an instance.

### 3.b Feature Extraction using PCA

In [738]:
table = Table()

max_depth = [2, 8, None]

parameters = {
  "table":table, "algorithm": "Baseline Decision Tree", "hyperparameter": "max_depth",
  "X":X_train_standardized, "Y":Y_train, "testX" : X_test_standardized, "testY": Y_test
}
for n in max_depth:
    trainAndPredictDecisionTree(**parameters, value=n, max_depth=n)

pca = PCA()
pca.fit(X_train_standardized)
X_train_pca = pca.transform(X_train_standardized)
X_test_pca = pca.transform(X_test_standardized)

parameters |= {
   "algorithm":"PCA Decision Tree",
   "X":X_train_pca, "Y":Y_train, "testX" : X_test_pca, "testY": Y_test
}
for n in max_depth:
    trainAndPredictDecisionTree(**parameters, value=n, max_depth=n)

table.tableFooter()



|Algorithm| Hyperparameter | Value | accuracy | precision | recall | f1 |
| --- | --- | --- | --- |--- | --- | --- |
| Baseline Decision Tree | max_depth | 2 | 0.883 | 0.8841 | 0.883 | 0.8815 |
|  |  | 8 | 0.9415 | 0.9445 | 0.9415 | 0.9407 |
|  |  | None | 0.9415 | 0.9445 | 0.9415 | 0.9407 |
| PCA Decision Tree | max_depth | 2 | 0.9123 | 0.9136 | 0.9123 | 0.9113 |
|  |  | 8 | 0.9415 | 0.9415 | 0.9415 | 0.9415 |
|  |  | None | 0.9415 | 0.9415 | 0.9415 | 0.9415 |


#### PCA Performance Evaluation

PCA performs similarly to the baseline tree. Both peak at the same values. However, PCA performs better at the minimum tree size. It might be interesting to see how PCA + AdaBoost perform. The baseline tree has a higher precision, indicating fewer false positives. The initial peak hints that since the features are orthogonal, more information is available earlier in the tree.

## 4. Comparison: PCA vs Feature Selection

In [739]:
table = Table()

parameters = {
  "table": table, "algorithm": "Decision Tree", "hyperparameter": "feature selection",
  "X":X_train_standardized, "Y":Y_train, "testX" : X_test_standardized, "testY": Y_test
}
trainAndPredictDecisionTree(**parameters, value="baseline", max_depth=None)


parameters |= {
    "algorithm": "Decision Tree",
    "X": X_train_filtered, "Y": Y_train, "testX": X_test_filtered, "testY": Y_test
}
trainAndPredictDecisionTree(**parameters, value="pearson", max_depth=None)

pca = PCA(n_components=len(pearson_filtered_columns))
pca.fit(X_train_standardized)
X_train_pca = pca.transform(X_train_standardized)
X_test_pca = pca.transform(X_test_standardized)
parameters |= {
    "algorithm": "Decision Tree",
    "X": X_train_pca, "Y": Y_train, "testX": X_test_pca, "testY": Y_test
}
trainAndPredictDecisionTree(**parameters, value="pca", max_depth=None)

table.tableFooter()


|Algorithm| Hyperparameter | Value | accuracy | precision | recall | f1 |
| --- | --- | --- | --- |--- | --- | --- |
| Decision Tree | feature selection | baseline | 0.9415 | 0.9445 | 0.9415 | 0.9407 |
|  |  | pearson | 0.9415 | 0.9419 | 0.9415 | 0.9412 |
|  |  | pca | 0.9357 | 0.9364 | 0.9357 | 0.9352 |


#### PCA vs Feature Selection Evaluation

PCA feature selection performed worse than Pearson or the baseline tree. While we saw that the PCA dimensions do encode the same information, this hints that the existing dimensions encode the information more tightly than PCA does. If there are "unused" features in the original dataset, PCA will use them for data. If those dimensions are then dropped, that will reduce the quality of the model.

Once the data is transformed using PCA, it is no longer interpretable. It is very difficult to take a path through the tree and match it back to features in the original dataset, since each PCA feature is multiple original features.

## 5. K-Fold Cross Validation

In [755]:
def report_score(score=None, score_accum=None):
    scores = score_accum.copy()
    scores = pd.concat([scores, score_accum.mean().rename("**Mean**").to_frame().T])
    scores = pd.concat([scores, score_accum.std().rename("**Std Dev**").to_frame().T])

    report(f"**{score}**")
    report(scores.to_markdown(floatfmt=".6f"))
    report("---")
    # report(score_accum.mean().rename("**Mean**").to_frame().T.to_markdown())
    # report(score_accum.std().rename("**Std Dev**").to_frame().T.to_markdown())

ignored = Table()

# Parameters shared by all callers.
parameters = {
  "cv":10,
  "table":ignored,
  "X":X_train, "Y":Y_train
}

report("**Step 1 - use normalised data**")
report("**Step 2 - create f1 and accuracy accumulators**")
f1_accum = pd.DataFrame()
accuracy_accum = pd.DataFrame()

# Setup for KNN
report("**Step 3 - run knn using cross_validate**")
parameters |= {
    "algorithm":"KNN", "hyperparameter":"n_neighbours"
}
knn_neighbours = [3,9]
for n in knn_neighbours:
    knn_scores = crossValidateUsingKnn(**parameters, value=n, n_neighbours=n)
    column_name = f"{parameters["algorithm"]}<br>(k={n})"
    accuracy_accum[column_name] = knn_scores["test_accuracy"]
    f1_accum[column_name] = knn_scores["test_f1_score"]

# Setup for Decision Tree
report("**Step 4 - run decision tree using cross_validate**")
parameters |= {
   "algorithm":"Decision Tree", "hyperparameter":"max_depth",
}
decision_tree_depths = [2,8]
for n in decision_tree_depths:
    dt_scores = crossValidateUsingTree(**parameters, value=n, max_depth=n)
    column_name = f"{parameters["algorithm"]}<br>(max_depth={n})"
    accuracy_accum[column_name] = dt_scores["test_accuracy"]
    f1_accum[column_name] = dt_scores["test_f1_score"]

report("**Step 5 - produce report**")
report_score("Accuracy", accuracy_accum)
report_score("F1 Score", f1_accum)



**Step 1 - use normalised data**

**Step 2 - create f1 and accuracy accumulators**

**Step 3 - run knn using cross_validate**

**Step 4 - run decision tree using cross_validate**

**Step 5 - produce report**

...

|             |   KNN<br>(k=3) |   KNN<br>(k=9) |   Decision Tree<br>(max_depth=2) |   Decision Tree<br>(max_depth=8) |
|:------------|---------------:|---------------:|---------------------------------:|---------------------------------:|
| 0           |       0.950000 |       0.925000 |                         0.850000 |                         0.825000 |
| 1           |       0.900000 |       0.925000 |                         0.800000 |                         0.875000 |
| 2           |       0.975000 |       0.975000 |                         0.900000 |                         0.950000 |
| 3           |       0.950000 |       0.950000 |                         0.950000 |                         0.900000 |
| 4           |       1.000000 |       0.975000 |                         0.875000 |                         0.925000 |
| 5           |       0.950000 |       0.975000 |                         0.950000 |                         0.975000 |
| 6           |       0.975000 |       0.975000 |                         0.925000 |                         0.925000 |
| 7           |       0.975000 |       0.950000 |                         0.900000 |                         0.950000 |
| 8           |       0.974359 |       0.974359 |                         0.897436 |                         1.000000 |
| 9           |       0.948718 |       1.000000 |                         0.948718 |                         0.923077 |
| **Mean**    |       0.959808 |       0.962436 |                         0.899615 |                         0.924808 |
| **Std Dev** |       0.026891 |       0.024260 |                         0.048452 |                         0.050004 |

---

**F1 Score**

|             |   KNN<br>(k=3) |   KNN<br>(k=9) |   Decision Tree<br>(max_depth=2) |   Decision Tree<br>(max_depth=8) |
|:------------|---------------:|---------------:|---------------------------------:|---------------------------------:|
| 0           |       0.950000 |       0.922545 |                         0.847009 |                         0.826302 |
| 1           |       0.898006 |       0.922545 |                         0.796011 |                         0.870909 |
| 2           |       0.974773 |       0.974773 |                         0.895238 |                         0.949003 |
| 3           |       0.949176 |       0.949176 |                         0.950000 |                         0.900000 |
| 4           |       1.000000 |       0.974814 |                         0.874070 |                         0.925444 |
| 5           |       0.950000 |       0.974814 |                         0.949176 |                         0.974814 |
| 6           |       0.974814 |       0.975148 |                         0.925444 |                         0.925444 |
| 7           |       0.974814 |       0.949176 |                         0.900000 |                         0.950000 |
| 8           |       0.974539 |       0.974539 |                         0.899387 |                         1.000000 |
| 9           |       0.948718 |       1.000000 |                         0.949359 |                         0.923618 |
| **Mean**    |       0.959484 |       0.961753 |                         0.898569 |                         0.924553 |
| **Std Dev** |       0.027393 |       0.025182 |                         0.049776 |                         0.050111 |

---

### Discuss Model Stablility and Reliability

KNN had a much more consistent result, with a standard deviation of 2.5 at K=9. The Decision Tree, with a standard deviation of ~5, had a much wider spread of results. This hints that KNN will generalize better than the Decision Tree.